# Day 25: SQL Joins & Aggregation

## Setup: Two Related Tables
Real databases are almost never one giant flat table — they're normalized into multiple smaller tables to avoid repeating data. Here, customer info lives in `customers` and order info lives in `orders`, linked by `customer_id`, which mirrors how a real e-commerce or CRM system would be structured: one customer can place many orders, but their name and city are stored only once. This separation avoids duplicating "Alice, NYC" every time she places a new order, but it means answering questions like "how much did each customer spend" requires *joining* the tables back together — which is exactly what the next few cells demonstrate. `executescript()` runs multiple SQL statements at once (creating both tables and inserting sample rows into each), and this cell produces no output — it just sets up the data.</cell id="cell-1">


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
# executescript runs multiple SQL statements in one call - creates both tables, then seeds them with rows
conn.executescript('''
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
    CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount INTEGER);
    INSERT INTO customers VALUES (1, 'Alice', 'NYC'), (2, 'Bob', 'LA'), (3, 'Charlie', 'NYC');
    INSERT INTO orders VALUES (1, 1, 200), (2, 1, 150), (3, 2, 300), (4, 3, 100);
''')

## INNER JOIN
A `JOIN` combines rows from two tables based on a shared key — here, matching each order to the customer who placed it via `customers.id = orders.customer_id`. By default, `JOIN` in SQL means `INNER JOIN`: only rows where the key matches in *both* tables are included, so a customer with zero orders (or an order with a bad customer_id) wouldn't appear in the result. This is the SQL equivalent of pandas' `pd.merge()`, and it's essential any time the information you need is spread across multiple tables — which is most of the time in a real database. The output pairs each customer's name with each of their order amounts, so a customer with two orders (like Alice) appears in two rows, one per order.</cell id="cell-3">


In [ ]:
query = '''
    SELECT customers.name, orders.amount
    FROM customers
    JOIN orders ON customers.id = orders.customer_id
'''
# JOIN (INNER JOIN) matches rows where customers.id = orders.customer_id; one output row per matched order
print(pd.read_sql_query(query, conn))

## GROUP BY with Aggregation
Joining tells you which order belongs to which customer, but you often want a summary rather than a row per order — total spend per city, count of orders per customer, and so on. `GROUP BY` collapses rows sharing the same value in a column into a single group, and an aggregate function like `SUM()` computes one summary value per group — this is the SQL equivalent of pandas' `groupby().sum()`. Combining a `JOIN` with `GROUP BY` is one of the most common patterns in real analytics SQL: join to bring related data together, then aggregate to summarize it. The output has one row per city, with `total_spent` equal to the sum of all order amounts from customers in that city (so NYC's total combines both Alice's and Charlie's orders).</cell id="cell-5">


In [ ]:
query = '''
    SELECT customers.city, SUM(orders.amount) AS total_spent
    FROM customers
    JOIN orders ON customers.id = orders.customer_id
    GROUP BY customers.city
'''
# GROUP BY city collapses matched rows into one group per city; SUM aggregates order amounts within each group
print(pd.read_sql_query(query, conn))

## HAVING
`WHERE` filters individual rows before any grouping happens, so it can't reference an aggregate result like `SUM(amount)` — at the point `WHERE` runs, that sum doesn't exist yet. `HAVING` solves this by filtering *after* `GROUP BY` and aggregation have already produced their summary values, letting you write conditions like "only show groups whose total exceeds some threshold." This distinction — WHERE filters rows, HAVING filters groups — is one of the most commonly confused parts of SQL for beginners, so it's worth internalizing early. In the output below, only customers whose total order amount exceeds 150 appear — customers whose combined orders are 150 or less are excluded from the result entirely.</cell id="cell-7">


In [ ]:
query = '''
    SELECT customer_id, SUM(amount) AS total
    FROM orders
    GROUP BY customer_id
    HAVING total > 150
'''
# HAVING filters groups AFTER aggregation (unlike WHERE, which filters rows before grouping)
print(pd.read_sql_query(query, conn))